# 00 — Buat Split Bersama (jalankan SEKALI)

Notebook ini membuat `splits.csv`: pembagian train/val/test **per `text_id`** untuk Stage 1 dan Stage 2.
Semua notebook training memakai file ini, sehingga:
- teks yang sama tidak pernah ada di train dan test sekaligus (tidak ada leakage di data explode), dan
- text-only dan metadata diuji pada **teks yang persis sama**.

Aturan pool & split sengaja dibuat sama dengan versi text-only sebelumnya (balancing 1:3 berdasarkan majority vote, `random_state=42`, test 20%, val 12,5% dari sisa).

Setelah selesai, upload `splits.csv` ke folder `DATA_DIR` (dataset Kaggle / `/content` di Colab).

In [ ]:
! pip install -q iterative-stratification

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [1]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

DATA_FILE = r"D:\I01620.Evelyn\indodiscourse\repo\thesis-uwuw\[USETHIS] Experiment Folder\data\data_thesis_full.csv"   # Colab: "/content/data_thesis_full.csv"
OUT_FILE  = r"D:\I01620.Evelyn\indodiscourse\repo\thesis-uwuw\[USETHIS] Experiment Folder\data\splits.csv"                                           # Colab: "/content/splits.csv"
SEED = 42

LABEL_COLS = ["insults", "identity_attack", "threat_incitement_to_violence",
              "profanity_obscenity", "sexually_explicit"]

In [2]:
df = pd.read_csv(DATA_FILE, dtype={"text_id": str})
assert df["text_id"].is_unique, "text_id tidak unik di data_thesis_full.csv"

cols = ["is_noise_or_spam_text", "toxicity", "polarized"] + LABEL_COLS
df_main = df.copy()
for col in cols:
    df_main[col] = df_main[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
vote_cols = [c for c in cols if c != "polarized"]

df_main["n_annotators"] = df_main["toxicity"].apply(len)

# Cek seri (seharusnya 0 setelah re-anotasi)
tie_report = {c: int(df_main[c].apply(lambda xs: np.mean([int(v) for v in xs]) == 0.5).sum()) for c in vote_cols}
print("Jumlah teks seri per kolom:", tie_report)

def majority_vote(x):
    return int(np.mean([int(i) for i in x]) > 0.5)   # identik dengan round(mean) sebelumnya; seri -> 0

for col in vote_cols:
    df_main[col] = df_main[col].apply(majority_vote)

df_main = df_main[df_main["is_noise_or_spam_text"] == 0]
df_main = df_main.dropna()
print("Teks setelah filter:", len(df_main))
print(df_main["toxicity"].value_counts())

Jumlah teks seri per kolom: {'is_noise_or_spam_text': 168, 'toxicity': 945, 'insults': 578, 'identity_attack': 513, 'threat_incitement_to_violence': 315, 'profanity_obscenity': 216, 'sexually_explicit': 45}
Teks setelah filter: 26379
toxicity
0    23980
1     2399
Name: count, dtype: int64


## Stage 1: balancing 1:3 lalu split per teks

In [3]:
pos = df_main[df_main["toxicity"] == 1]
neg = df_main[df_main["toxicity"] == 0].sample(n=3 * len(pos), random_state=SEED)
df_final = pd.concat([pos, neg]).sample(frac=1, random_state=SEED).reset_index(drop=True)

ids, y = df_final["text_id"], df_final["toxicity"]
ids_train_full, ids_test, y_train_full, _ = train_test_split(ids, y, test_size=0.2, stratify=y, random_state=SEED)
ids_train, ids_val, _, _ = train_test_split(ids_train_full, y_train_full, test_size=0.125,
                                            stratify=y_train_full, random_state=SEED)
s1 = pd.concat([pd.DataFrame({"text_id": ids_train, "s1_split": "train"}),
                pd.DataFrame({"text_id": ids_val, "s1_split": "val"}),
                pd.DataFrame({"text_id": ids_test, "s1_split": "test"})])
print(s1["s1_split"].value_counts())

s1_split
train    6716
test     1920
val       960
Name: count, dtype: int64


## Stage 2: teks toxic (majority), split multi-label per teks

In [4]:
df_toxic = df_main[df_main["toxicity"] == 1].reset_index(drop=True)

def ml_split(x, y, test_size):
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=SEED)
    tr, te = next(msss.split(x, y.values))
    return x.iloc[tr], x.iloc[te], y.iloc[tr], y.iloc[te]

x2, y2 = df_toxic["text_id"], df_toxic[LABEL_COLS]
x2_train_full, x2_test, y2_train_full, y2_test = ml_split(x2, y2, 0.2)
x2_train, x2_val, y2_train, y2_val = ml_split(x2_train_full, y2_train_full, 0.125)
s2 = pd.concat([pd.DataFrame({"text_id": x2_train, "s2_split": "train"}),
                pd.DataFrame({"text_id": x2_val, "s2_split": "val"}),
                pd.DataFrame({"text_id": x2_test, "s2_split": "test"})])
print(s2["s2_split"].value_counts())
print(pd.DataFrame({"train": y2_train.sum(), "val": y2_val.sum(), "test": y2_test.sum()}))

s2_split
train    1679
test      480
val       240
Name: count, dtype: int64
                               train  val  test
insults                          738  106   211
identity_attack                  696   99   199
threat_incitement_to_violence     75   11    22
profanity_obscenity              251   36    72
sexually_explicit                 43    6    12


In [5]:
splits = (df_main[["text_id", "toxicity", "n_annotators"]]
          .rename(columns={"toxicity": "toxicity_majority"})
          .merge(s1, on="text_id", how="left")
          .merge(s2, on="text_id", how="left"))
splits = splits[splits["s1_split"].notna() | splits["s2_split"].notna()]

for col in ["s1_split", "s2_split"]:
    g = splits.dropna(subset=[col])
    assert g["text_id"].is_unique
print(splits.groupby("s1_split")["n_annotators"].describe())
splits.to_csv(OUT_FILE, index=False)
print("Tersimpan:", OUT_FILE, "|", len(splits), "teks")

           count      mean       std  min  25%  50%  75%   max
s1_split                                                      
test      1920.0  2.123438  1.972154  1.0  1.0  1.0  3.0  19.0
train     6716.0  2.143985  2.059349  1.0  1.0  1.0  3.0  19.0
val        960.0  2.186458  2.000945  1.0  1.0  2.0  3.0  19.0
Tersimpan: D:\I01620.Evelyn\indodiscourse\repo\thesis-uwuw\[USETHIS] Experiment Folder\data\splits.csv | 9596 teks
